# 06 — Propensity Scores and Matching
Overlap, propensity estimation, matching and balance

**Data are synthetic.** Run from repository root or upload the CSV file to Colab.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

def load_csv(name):
    candidates=[Path('data')/name, Path('../data')/name, Path('/content')/name]
    for p in candidates:
        if p.exists():
            return pd.read_csv(p)
    raise FileNotFoundError(f'Upload {name} in Colab, or run from repository root.')

df=load_csv('campaign_observational.csv')
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
X=df[['age','prior_spend','loyalty']];model=LogisticRegression(max_iter=2000).fit(X,df.treated)
df['ps']=model.predict_proba(X)[:,1];treated=df[df.treated==1];control=df[df.treated==0]
nn=NearestNeighbors(n_neighbors=1).fit(control[['ps']]);dist,idx=nn.kneighbors(treated[['ps']]);caliper=.05
keep=dist[:,0]<=caliper
matched_control=control.iloc[idx[keep,0]];matched_treated=treated.iloc[np.flatnonzero(keep)]
print('Pairs within caliper:',len(matched_treated));print('ATT-style matched difference:',(matched_treated.post_spend.to_numpy()-matched_control.post_spend.to_numpy()).mean())
print('Matching with replacement; assess covariate balance and overlap before interpretation.')

## Reflection
What design assumption is most important here? How could it fail in real data?
